# โหลด Tool ทั้งหมดที่จะใช้

## การติดตั้ง Dependencies
ในเซลล์แรก เราติดตั้ง libraries ที่จำเป็น:
- **transformers**: สำหรับโหลดและใช้งาน pre-trained models
- **torch**: Deep learning framework ที่ต้องใช้กับ transformers
- **sentencepiece**: Tokenizer library สำหรับการประมวลผลภาษา

## Pipeline ที่ใช้

### 1. **NER Pipeline** (Named Entity Recognition)
ใช้โมเดล `pythainlp/thainer-corpus-v2-base-model` สำหรับ:
- ระบุคน (PERSON)
- ระบุองค์กร (ORGANIZATION)
- ระบุสถานที่ (LOCATION)
- กรองเฉพาะ entities ที่มี confidence score > 0.85

### 2. **Sentiment Analysis** (Rule-based)
วิเคราะห์ความรู้สึกจากข้อความโดย:
- นับคำบ่งชี้ด้านบวก (positive keywords)
- นับคำบ่งชี้ด้านลบ (negative keywords)
- เปรียบเทียบจำนวนเพื่อกำหนด sentiment

## ผลลัพธ์
- **NER Results**: รายชื่อบุคคลและองค์กรที่พบในข้อความ
- **Sentiment**: ความรู้สึก (positive/negative/neutral) พร้อมคะแนน
- **Output**: บันทึก CSV ที่ประมวลผลแล้ว และสรุปข้อมูลสถิติ

In [ ]:
pip install transformers torch sentencepiece

# เทสโมเดล

In [3]:
# test_ner.py
from transformers import pipeline

ner = pipeline(
    task='ner',
    model='pythainlp/thainer-corpus-v2-base-model',
    aggregation_strategy='simple'
)

test_text = "ดร.เทดรอส องค์การอนามัยโลก ยืนยันว่าไม่พบการระบาดของไวรัสฮันตา"
result = ner(test_text)
print(result)

c:\Users\Radeon\anaconda3\envs\test\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Radeon\.cache\huggingface\hub\models--pythainlp--thainer-corpus-v2-base-model. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back

[{'entity_group': 'PERSON', 'score': 0.989648, 'word': 'ดร.เทดรอส', 'start': 0, 'end': 9}, {'entity_group': 'ORGANIZATION', 'score': 0.99186486, 'word': '', 'start': 9, 'end': 10}, {'entity_group': 'ORGANIZATION', 'score': 0.99266773, 'word': 'องค์การ', 'start': 10, 'end': 17}, {'entity_group': 'ORGANIZATION', 'score': 0.720488, 'word': 'อนามัยโลก', 'start': 17, 'end': 26}]


### 02_ner_sentiment

In [6]:
# 02_ner_sentiment.py
import pandas as pd
from transformers import pipeline
from collections import Counter

# โหลด model ครั้งเดียว
print("Loading NER model...")
ner = pipeline(
    task='ner',
    model='pythainlp/thainer-corpus-v2-base-model',
    aggregation_strategy='simple'
)
print("Model loaded!")

df = pd.read_csv("D:\\Ma work\\project\\data\\articles.csv", encoding="utf-8-sig")
print(f"Loaded {len(df)} articles")

# --- NER: ดึงเฉพาะ PERSON ---
def extract_persons(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    try:
        # model รับได้สูงสุด ~512 tokens — ตัด text ยาวเกินก่อน
        entities = ner(text[:1000])
        persons = [
            e['word'].strip()
            for e in entities
            if e['entity_group'] == 'PERSON'
            and len(e['word'].strip()) > 1
            and e['score'] > 0.85   # กรอง low confidence ออก
        ]
        return persons
    except Exception as ex:
        print(f"NER error: {ex}")
        return []

# --- Sentiment: rule-based เหมือนเดิม ---
POSITIVE = [
    'ชนะ', 'ประสบความสำเร็จ', 'เยี่ยม', 'โดดเด่น', 'ยกย่อง',
    'ชื่นชม', 'ภูมิใจ', 'สำเร็จ', 'ได้รับการยอมรับ', 'แก้ปัญหาได้',
    'ฟื้นตัว', 'ก้าวหน้า', 'บรรลุ', 'ยินดี', 'ดีใจ'
]
NEGATIVE = [
    'แพ้', 'วิจารณ์', 'โจมตี', 'เสียหาย', 'ล้มเหลว', 'ต่อต้าน',
    'ทุจริต', 'ฉ้อโกง', 'ถูกจับ', 'ลาออก', 'ไล่ออก', 'ประท้วง',
    'ขัดแย้ง', 'เสียชีวิต', 'อุบัติเหตุ', 'ระเบิด', 'โกง', 'คดี',
    'ระบาด', 'เตือน', 'อันตราย', 'วิกฤต', 'ถูกฟ้อง', 'จับกุม'
]

def get_sentiment(text):
    if not isinstance(text, str):
        return 'neutral', 0, 0
    pos = sum(1 for w in POSITIVE if w in text)
    neg = sum(1 for w in NEGATIVE if w in text)
    if pos > neg:
        return 'positive', pos, neg
    elif neg > pos:
        return 'negative', pos, neg
    return 'neutral', pos, neg

# --- ประมวลผล ---
print("Processing NER... (อาจใช้เวลา 2-3 นาที)")
df['persons']      = df['text'].apply(extract_persons)
df['person_count'] = df['persons'].apply(len)

sentiments         = df['text'].apply(get_sentiment)
df['sentiment']    = sentiments.apply(lambda x: x[0])
df['pos_score']    = sentiments.apply(lambda x: x[1])
df['neg_score']    = sentiments.apply(lambda x: x[2])

df.to_csv("data/articles_processed.csv", index=False, encoding="utf-8-sig")

# --- Summary ---
print("\n=== Results ===")
print(df[['title', 'sentiment', 'pos_score', 'neg_score', 'person_count']].to_string())

print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

all_persons = [p for sublist in df['persons'] for p in sublist]
print("\nTop 10 persons mentioned:")
for name, count in Counter(all_persons).most_common(10):
    print(f"  {name}: {count}")

Loading NER model...


Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Model loaded!
Loaded 10 articles
Processing NER... (อาจใช้เวลา 2-3 นาที)

=== Results ===
                                                                                                               title sentiment  pos_score  neg_score  person_count
0                                       ผอ. WHO ชี้ ไม่พบสัญญาณ การระบาดใหญ่ของไวรัสฮันตา Logo Thairath สมาชิก ค้นหา  negative          0          3             2
1                   ราชกิจจาฯ ประกาศ 8 พื้นที่ ห้ามขาย-ดื่ม เหล้าเบียร์ มีผลตั้งแต่วันนี้ Logo Thairath สมาชิก ค้นหา   neutral          0          0             0
2  ผอ.เอฟบีไอ โต้เดือดกลางที่ประชุมวุฒิสภา หลังถูกกล่าวหาว่าดื่มหนักระหว่างปฏิบัติหน้าที่ Logo Thairath สมาชิก ค้นหา  negative          1          2             6
3     ชาวบ้านบุกโรงงานสารเคมี จ.ขอนแก่น ร้องตรวจสอบปมปล่อยน้ำเสีย ลงสู่ที่นา นานกว่า 4 ปี Logo Thairath สมาชิก ค้นหา  negative          0          1             2
4                                                             ‘เอลนีโญ’ วิกฤตลูกใหม่เศรษฐกิจไทย